In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.abspath('..'))
from utils import (
    avg_prf_at_k,
    avg_mrr_at_k,
    avg_hits_at_k,
    avg_jaccard_at_k,
    avg_ndcg_at_k,
)

MINILM_MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 32
TOPKS = [3, 5, 10, 20]


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = "../dataset/hard_triplets_tfidf_dbpedia.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Successfully loaded {len(data)} query groups from the JSON file.")


Successfully loaded 337 query groups from the JSON file.


In [3]:
unique_queries = set()
eval_items = []

for item in data:
    anchor = item["anchor_query"]
    positives = item["positive_queries"]
    negatives = item["negative_queries"]

    pool = positives + negatives

    unique_queries.add(anchor)
    unique_queries.update(positives)
    unique_queries.update(negatives)

    eval_items.append((anchor, pool, set(positives)))

queries_list = list(unique_queries)
print(f"Loaded {len(eval_items)} evaluation items.")
print(f"Unique queries in vocabulary: {len(queries_list)}")


Loaded 337 evaluation items.
Unique queries in vocabulary: 1677


In [4]:
# Binary relevance: 1.0 if candidate is in positive_queries, else 0.0.
# No SPARQL endpoint, no cache, and no safe_jaccard_for_queries needed.

relevance_scores_per_item = []

for _, pool, positives in eval_items:
    scores = {cand: (1.0 if cand in positives else 0.0) for cand in pool}
    relevance_scores_per_item.append(scores)

print(f"Binary relevance scores built for {len(relevance_scores_per_item)} items. No SPARQL calls needed.")


Binary relevance scores built for 337 items. No SPARQL calls needed.


In [5]:
def build_embedding_map(model, queries, batch_size=32, normalize_embeddings=True):
    """Encodes all queries exactly once and stores them in a dict."""
    embedding_map = {}
    for start in range(0, len(queries), batch_size):
        batch = queries[start:start + batch_size]
        embeddings = model.encode(
            batch,
            convert_to_numpy=True,
            normalize_embeddings=normalize_embeddings,
        )
        for query, emb in zip(batch, embeddings):
            embedding_map[query] = emb
    return embedding_map


def evaluate_dense_model(model_name, model, queries, items, batch_size=32):
    """Encodes all queries once, then ranks each pool by dot-product similarity."""
    print(f"\nEvaluating dense model: {model_name}")

    embedding_map = build_embedding_map(model, queries, batch_size)
    sorted_pools = []

    for anchor_query, pool, _ in items:
        pool_embs = np.vstack([embedding_map[q] for q in pool])
        anchor_emb = embedding_map[anchor_query]
        scores = pool_embs @ anchor_emb
        order = np.argsort(-scores)
        sorted_pools.append([pool[i] for i in order])

    return sorted_pools


In [6]:
print(f"Loading {MINILM_MODEL_NAME}...")
minilm_model = SentenceTransformer(MINILM_MODEL_NAME)

minilm_sorted_pools = evaluate_dense_model(
    model_name=MINILM_MODEL_NAME,
    model=minilm_model,
    queries=list(unique_queries),
    items=eval_items,
    batch_size=BATCH_SIZE,
)

print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS: Baseline MiniLM")
print("=" * 60)

print("\n[Binary relevance: Annotated Positives]")
print(f"{'':5} {'P':>7} {'R':>7} {'F1':>7} {'MRR':>7} {'Hits':>7} {'avgJacc':>9} {'nDCG':>7}")
print("-" * 62)
for k in TOPKS:
    p, r, f1 = avg_prf_at_k(minilm_sorted_pools, eval_items, k)
    mrr = avg_mrr_at_k(minilm_sorted_pools, eval_items, k)
    hits = avg_hits_at_k(minilm_sorted_pools, eval_items, k)
    avg_jacc = avg_jaccard_at_k(minilm_sorted_pools, k, relevance_scores_per_item)
    ndcg = avg_ndcg_at_k(minilm_sorted_pools, k, relevance_scores_per_item)
    print(f"@{k:<4} {p:>7.4f} {r:>7.4f} {f1:>7.4f} {mrr:>7.4f} {hits:>7.4f} {avg_jacc:>9.4f} {ndcg:>7.4f}")


Loading all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5801.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Evaluating dense model: all-MiniLM-L6-v2

FINAL EVALUATION RESULTS: Baseline MiniLM

[Binary relevance: Annotated Positives]
            P       R      F1     MRR    Hits   avgJacc    nDCG
--------------------------------------------------------------
@3     0.3818  0.1524  0.1975  0.4075  0.5104    0.3818  0.3696
@5     0.3757  0.2411  0.2624  0.4185  0.5579    0.3757  0.3891
@10    0.3546  0.4182  0.3422  0.4327  0.6677    0.3546  0.4362
@20    0.3310  0.8123  0.4219  0.4576  0.9941    0.3310  0.5784
